# Computational Notebook 13: Stablecoin Analysis

## Overview

Stablecoins are cryptocurrencies designed to maintain a stable value relative to a reference asset, typically the US dollar. They serve as the primary medium of exchange in DeFi, bridging volatile crypto markets with stable purchasing power. This notebook explores the three major stablecoin architectures -- fiat-collateralized, crypto-collateralized, and algorithmic -- through working simulations. We build a Collateralized Debt Position (CDP) system modeled on MakerDAO, simulate algorithmic stablecoin dynamics including death spirals (Terra/LUNA case study), analyze peg stability mechanics, and evaluate risk metrics for stablecoin portfolios.

## Prerequisites
- **Notebook 05**: DeFi Protocols (AMMs, lending, liquidation mechanics)
- **Notebook 08**: Valuation Models (DCF, relative valuation)
- Basic Python programming and familiarity with NumPy

## Learning Objectives

1. Classify stablecoins by collateral mechanism and understand tradeoffs of each design
2. Simulate peg arbitrage mechanics that maintain fiat-collateralized stablecoin prices
3. Build a CDP system with collateralization ratios, stability fees, and liquidation
4. Model algorithmic stablecoin mint/burn dynamics and simulate death spiral scenarios
5. Calculate stablecoin risk metrics: peg deviation, collateralization health, concentration risk
6. Analyze yield sources for stablecoins and compare risk-adjusted returns

**Estimated Time:** 4-6 hours

**Related Content:** [Section 07: Stablecoins](../sections/07-stablecoins.md)

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

print("All imports successful!")
print("This notebook simulates stablecoin mechanisms and risk analysis.")

---
## 1. Stablecoin Taxonomy

Stablecoins can be classified by their stabilization mechanism:

| Type | Examples | Collateral | Peg Mechanism | Trust Assumption |
|------|----------|------------|---------------|------------------|
| Fiat-collateralized | USDT, USDC | USD reserves | 1:1 redemption | Custodian honesty |
| Crypto-collateralized | DAI, LUSD | ETH, BTC | Over-collateralization + liquidation | Smart contract security |
| Algorithmic | UST (defunct), FRAX | None or partial | Mint/burn with seigniorage token | Market confidence |

> **Definition: Seigniorage** -- The profit earned by a currency issuer, calculated as the difference between the face value of money and its production cost. In algorithmic stablecoins, seigniorage accrues to holders of the companion governance/equity token.

> **Definition: Collateralized Debt Position (CDP)** -- A smart contract position where a user locks collateral (e.g., ETH) to mint stablecoins (e.g., DAI). The collateral must exceed the minted value by a safety margin (e.g., 150%).

**Source:** Moin, A. et al. (2020). "SoK: A Classification Framework for Stablecoin Designs." *FC 2020*.

In [ ]:
# Stablecoin market overview (synthetic but realistic data)
@dataclass
class StablecoinProfile:
    """Profile of a stablecoin."""
    name: str
    ticker: str
    type: str
    market_cap_b: float
    collateral_ratio: float  # 1.0 for fiat, >1 for crypto, <1 for algo
    avg_peg_deviation_bps: float  # Average deviation in basis points
    max_depeg_pct: float  # Worst historical depeg
    launch_year: int


stablecoins = [
    StablecoinProfile("Tether", "USDT", "Fiat", 95.0, 1.0, 5, 5.0, 2014),
    StablecoinProfile("USD Coin", "USDC", "Fiat", 32.0, 1.0, 2, 12.0, 2018),
    StablecoinProfile("Dai", "DAI", "Crypto", 5.0, 1.50, 8, 8.0, 2019),
    StablecoinProfile("LUSD", "LUSD", "Crypto", 0.5, 1.10, 15, 5.0, 2021),
    StablecoinProfile("FRAX", "FRAX", "Hybrid", 1.0, 0.92, 10, 6.0, 2020),
    StablecoinProfile("TerraUSD", "UST", "Algo", 0.0, 0.0, 50, 99.9, 2020),
]

print("=" * 80)
print("STABLECOIN MARKET OVERVIEW")
print("=" * 80)

print(f"\n{'Name':<14} {'Ticker':<7} {'Type':<8} {'MCap':>8} {'CR':>6} {'Avg Dev':>9} {'Max Depeg':>11}")
print("-" * 70)
for s in stablecoins:
    status = '(defunct)' if s.market_cap_b == 0 else ''
    print(f"{s.name:<14} {s.ticker:<7} {s.type:<8} ${s.market_cap_b:>6.1f}B "
          f"{s.collateral_ratio:>5.0%} {s.avg_peg_deviation_bps:>7.0f}bps "
          f"{s.max_depeg_pct:>9.1f}% {status}")

total_mcap = sum(s.market_cap_b for s in stablecoins)
print(f"\nTotal stablecoin market cap: ${total_mcap:.1f}B")
print(f"Fiat-backed dominance: {sum(s.market_cap_b for s in stablecoins if s.type == 'Fiat') / total_mcap * 100:.1f}%")

---
## 2. Peg Stability Mechanics

Fiat-collateralized stablecoins maintain their peg through arbitrage:

- **Price > $1.00**: Arbitrageurs mint new stablecoins (deposit $1, receive 1 token) and sell on market for profit
- **Price < $1.00**: Arbitrageurs buy on market (below $1) and redeem for $1 from the issuer

This creates a negative feedback loop that pulls the price back to $1.00.

For crypto-collateralized stablecoins, the mechanism is different:
- **Price > $1.00**: Users open CDPs to mint new DAI (cheap to create) and sell at premium
- **Price < $1.00**: Users buy DAI cheaply and repay their CDP debt at face value

In [ ]:
def simulate_peg_arbitrage(initial_price: float, peg: float = 1.0,
                           arb_strength: float = 0.3,
                           noise_std: float = 0.002,
                           steps: int = 200) -> np.ndarray:
    """Simulate stablecoin price with arbitrage peg mechanism.
    
    Args:
        initial_price: Starting price
        peg: Target price
        arb_strength: How quickly arbitrage corrects deviations (0-1)
        noise_std: Standard deviation of random market noise
        steps: Number of time steps
    
    Returns:
        Array of prices over time
    """
    prices = np.zeros(steps)
    prices[0] = initial_price
    
    for t in range(1, steps):
        # Mean reversion from arbitrage
        deviation = prices[t-1] - peg
        arb_force = -arb_strength * deviation
        
        # Random market noise
        noise = np.random.normal(0, noise_std)
        
        prices[t] = prices[t-1] + arb_force + noise
    
    return prices


# Simulate different arbitrage strengths
np.random.seed(42)

print("=" * 60)
print("PEG ARBITRAGE SIMULATION")
print("=" * 60)

scenarios = [
    ("Strong arb (USDC-like)", 0.5, 0.001),
    ("Moderate arb (DAI-like)", 0.2, 0.003),
    ("Weak arb (algo-like)", 0.05, 0.005),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (label, strength, noise) in zip(axes, scenarios):
    # Start with a shock above peg
    prices = simulate_peg_arbitrage(1.02, arb_strength=strength, noise_std=noise)
    ax.plot(prices, 'b-', alpha=0.8)
    ax.axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Peg')
    ax.fill_between(range(len(prices)), 0.995, 1.005, alpha=0.1, color='green')
    ax.set_title(label)
    ax.set_xlabel('Time Steps')
    ax.set_ylabel('Price ($)')
    ax.set_ylim(0.97, 1.05)
    ax.legend(fontsize=8)
    
    dev = np.abs(prices - 1.0)
    print(f"{label}: Mean dev={np.mean(dev)*100:.3f}%, Max dev={np.max(dev)*100:.3f}%")

plt.tight_layout()
plt.savefig('/tmp/peg_arbitrage.png', dpi=100, bbox_inches='tight')
plt.show()
print("Stronger arbitrage mechanisms produce tighter pegs.")

---
## 3. CDP (Collateralized Debt Position) Simulation

MakerDAO pioneered the CDP model for creating crypto-collateralized stablecoins:

1. User deposits ETH into a CDP (called a "Vault" in Maker V2)
2. User mints DAI up to the collateralization limit (e.g., 66% of collateral value at 150% ratio)
3. User pays a **stability fee** (annual interest) on the minted DAI
4. If collateral ratio falls below the **liquidation ratio**, the position is liquidated

$$\text{Collateralization Ratio} = \frac{\text{Collateral Value}}{\text{Debt}} \times 100\%$$

$$\text{Max Mintable} = \frac{\text{Collateral Value}}{\text{Min Collateral Ratio}}$$

**Source:** MakerDAO. (2023). "Introduction to the Maker Protocol." docs.makerdao.com

In [ ]:
@dataclass
class Vault:
    """A CDP vault."""
    owner: str
    collateral_amount: float   # Amount of collateral (e.g., ETH)
    debt: float                # Amount of stablecoin minted
    created_at: int = 0
    liquidated: bool = False


class CDPSystem:
    """MakerDAO-style CDP (Collateralized Debt Position) system."""
    
    def __init__(self, collateral_name: str = "ETH",
                 stablecoin_name: str = "DAI",
                 min_collateral_ratio: float = 1.50,
                 liquidation_ratio: float = 1.50,
                 liquidation_penalty: float = 0.13,
                 stability_fee: float = 0.05) -> None:
        """Initialize CDP system.
        
        Args:
            min_collateral_ratio: Minimum ratio to open/maintain vault
            liquidation_ratio: Ratio below which vault is liquidated
            liquidation_penalty: Penalty applied during liquidation (13%)
            stability_fee: Annual interest rate on minted stablecoins
        """
        self.collateral_name = collateral_name
        self.stablecoin_name = stablecoin_name
        self.min_cr = min_collateral_ratio
        self.liq_ratio = liquidation_ratio
        self.liq_penalty = liquidation_penalty
        self.stability_fee = stability_fee
        self.collateral_price = 2000.0
        self.vaults: Dict[int, Vault] = {}
        self.next_id = 1
        self.total_debt = 0.0
        self.total_collateral = 0.0
        self.liquidation_log: List[Dict] = []
    
    def open_vault(self, owner: str, collateral: float, debt: float) -> int:
        """Open a new CDP vault."""
        collateral_value = collateral * self.collateral_price
        cr = collateral_value / debt if debt > 0 else float('inf')
        
        if cr < self.min_cr:
            raise ValueError(
                f"CR {cr:.2f} below minimum {self.min_cr:.2f}. "
                f"Max debt: ${collateral_value / self.min_cr:,.2f}")
        
        vid = self.next_id
        self.next_id += 1
        self.vaults[vid] = Vault(owner, collateral, debt)
        self.total_debt += debt
        self.total_collateral += collateral
        return vid
    
    def get_cr(self, vault_id: int) -> float:
        """Get current collateralization ratio."""
        v = self.vaults[vault_id]
        if v.debt == 0:
            return float('inf')
        return (v.collateral_amount * self.collateral_price) / v.debt
    
    def update_price(self, new_price: float) -> List[Dict]:
        """Update collateral price and check for liquidations."""
        self.collateral_price = new_price
        liquidated = []
        
        for vid, vault in self.vaults.items():
            if vault.liquidated or vault.debt == 0:
                continue
            cr = self.get_cr(vid)
            if cr < self.liq_ratio:
                # Liquidate
                collateral_value = vault.collateral_amount * new_price
                debt_to_cover = vault.debt
                penalty = debt_to_cover * self.liq_penalty
                total_cost = debt_to_cover + penalty
                
                # Collateral seized
                collateral_seized = min(vault.collateral_amount,
                                       total_cost / new_price)
                collateral_returned = vault.collateral_amount - collateral_seized
                
                vault.liquidated = True
                self.total_debt -= vault.debt
                self.total_collateral -= vault.collateral_amount
                
                result = {
                    "vault_id": vid,
                    "owner": vault.owner,
                    "cr_at_liquidation": cr,
                    "debt": debt_to_cover,
                    "collateral_seized": collateral_seized,
                    "collateral_returned": collateral_returned,
                    "penalty": penalty,
                    "price": new_price
                }
                liquidated.append(result)
                self.liquidation_log.append(result)
        
        return liquidated
    
    def system_stats(self) -> Dict[str, float]:
        """Get system-wide statistics."""
        active = [v for v in self.vaults.values() if not v.liquidated and v.debt > 0]
        if not active:
            return {"active_vaults": 0, "total_debt": 0, "total_collateral_value": 0,
                    "system_cr": 0, "min_cr": 0, "median_cr": 0}
        
        crs = [(v.collateral_amount * self.collateral_price) / v.debt for v in active]
        return {
            "active_vaults": len(active),
            "total_debt": self.total_debt,
            "total_collateral_value": self.total_collateral * self.collateral_price,
            "system_cr": (self.total_collateral * self.collateral_price) / self.total_debt if self.total_debt > 0 else 0,
            "min_cr": min(crs),
            "median_cr": float(np.median(crs))
        }


# Demonstrate CDP system
cdp = CDPSystem()

print("=" * 60)
print("CDP SYSTEM SIMULATION (MakerDAO-style)")
print("=" * 60)
print(f"ETH price: ${cdp.collateral_price:,.0f}")
print(f"Min collateral ratio: {cdp.min_cr:.0%}")
print(f"Liquidation penalty: {cdp.liq_penalty:.0%}")

# Open several vaults with different risk levels
np.random.seed(42)
vault_configs = [
    ("Conservative", 10, 8000),     # 250% CR
    ("Moderate", 10, 10000),        # 200% CR
    ("Aggressive", 10, 12000),      # 167% CR
    ("Risky", 10, 13000),           # 154% CR
]

print(f"\n{'Vault':<14} {'Collateral':>12} {'Debt':>10} {'CR':>8}")
print("-" * 48)
vault_ids = []
for label, coll, debt in vault_configs:
    vid = cdp.open_vault(label, coll, debt)
    vault_ids.append(vid)
    cr = cdp.get_cr(vid)
    print(f"{label:<14} {coll:>10} ETH ${debt:>8,} {cr:>7.0%}")

stats = cdp.system_stats()
print(f"\nSystem CR: {stats['system_cr']:.0%}")
print(f"Total DAI minted: ${stats['total_debt']:,.0f}")

In [ ]:
# Simulate ETH price crash and cascading liquidations
print("=" * 60)
print("CASCADING LIQUIDATION SIMULATION")
print("=" * 60)

# Reset CDP system with many vaults
np.random.seed(42)
cdp2 = CDPSystem()
n_vaults = 200

# Create vaults with varying CRs (150%-300%)
for i in range(n_vaults):
    coll = np.random.uniform(5, 50)
    target_cr = np.random.uniform(1.55, 3.0)
    debt = (coll * cdp2.collateral_price) / target_cr
    cdp2.open_vault(f"user_{i:03d}", coll, debt)

# Price crash simulation
prices = np.linspace(2000, 800, 50)
price_history = []
debt_history = []
cr_history = []
liq_count_history = []
total_liquidated = 0

for price in prices:
    liqs = cdp2.update_price(price)
    total_liquidated += len(liqs)
    stats = cdp2.system_stats()
    
    price_history.append(price)
    debt_history.append(stats['total_debt'])
    cr_history.append(stats['system_cr'] if stats['system_cr'] > 0 else None)
    liq_count_history.append(total_liquidated)

print(f"Started with {n_vaults} vaults")
print(f"Total liquidated: {total_liquidated}")
print(f"Remaining active: {n_vaults - total_liquidated}")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(price_history, 'r-', linewidth=2)
axes[0, 0].set_ylabel('ETH Price ($)')
axes[0, 0].set_title('ETH Price Crash')

axes[0, 1].plot(liq_count_history, 'orange', linewidth=2)
axes[0, 1].set_ylabel('Cumulative Liquidations')
axes[0, 1].set_title('Cascading Liquidations')

axes[1, 0].plot([d/1e6 for d in debt_history], 'b-', linewidth=2)
axes[1, 0].set_ylabel('Total DAI Outstanding ($M)')
axes[1, 0].set_title('System Debt Shrinkage')
axes[1, 0].set_xlabel('Time Step')

valid_cr = [(i, cr) for i, cr in enumerate(cr_history) if cr is not None and cr < 10]
if valid_cr:
    axes[1, 1].plot([x[0] for x in valid_cr], [x[1] for x in valid_cr], 'g-', linewidth=2)
    axes[1, 1].axhline(y=1.5, color='red', linestyle='--', label='Liquidation ratio')
    axes[1, 1].set_ylabel('System Collateralization Ratio')
    axes[1, 1].set_title('System Health')
    axes[1, 1].set_xlabel('Time Step')
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig('/tmp/cdp_liquidation.png', dpi=100, bbox_inches='tight')
plt.show()
print("Liquidations accelerate as price drops -- each liquidation sells collateral,")
print("potentially pushing price further down (liquidation cascade).")

---
## 4. Algorithmic Stablecoin Dynamics

Algorithmic stablecoins attempt to maintain their peg without full collateral backing, using a companion token to absorb volatility.

### Seigniorage Shares Model
- **Price > $1**: Protocol mints new stablecoins, selling them to push price down. Revenue goes to equity token holders.
- **Price < $1**: Protocol buys and burns stablecoins using newly minted equity tokens. This dilutes equity holders but restores the peg.

### The Death Spiral
When confidence collapses:
1. Stablecoin drops below peg
2. Protocol mints equity tokens to buy back stablecoins
3. Equity token price crashes due to dilution
4. Users lose confidence and sell more stablecoins
5. More equity must be minted -> more dilution -> repeat

This is exactly what happened to Terra/LUNA in May 2022, wiping out ~$40 billion.

**Source:** Clements, R. (2021). "Built to Fail: The Inherent Fragility of Algorithmic Stablecoins." *Wake Forest Law Review*.

In [ ]:
class AlgorithmicStablecoin:
    """Seigniorage shares algorithmic stablecoin simulator."""
    
    def __init__(self, stable_supply: float = 10_000_000_000,
                 equity_supply: float = 1_000_000_000,
                 equity_price: float = 40.0,
                 peg: float = 1.0) -> None:
        """Initialize algorithmic stablecoin."""
        self.stable_supply = stable_supply
        self.equity_supply = equity_supply
        self.equity_price = equity_price
        self.peg = peg
        self.stable_price = peg
        self.equity_mcap = equity_supply * equity_price
        self.history = []
        self._record()
    
    def _record(self) -> None:
        """Record current state."""
        self.history.append({
            'stable_price': self.stable_price,
            'stable_supply': self.stable_supply,
            'equity_price': self.equity_price,
            'equity_supply': self.equity_supply,
            'equity_mcap': self.equity_mcap,
            'stable_mcap': self.stable_supply * self.stable_price
        })
    
    def step(self, demand_shock: float = 0, confidence: float = 1.0) -> None:
        """Simulate one time step.
        
        Args:
            demand_shock: Change in stablecoin demand (negative = selling)
            confidence: Market confidence in the peg (0-1)
        """
        # Apply demand shock to stable price
        price_impact = demand_shock / self.stable_supply * 10  # Simplified
        self.stable_price = max(0.001, self.stable_price + price_impact)
        
        # Protocol response
        if self.stable_price > self.peg * 1.005:  # Above peg
            # Mint and sell stablecoins
            mint_amount = self.stable_supply * (self.stable_price / self.peg - 1) * 0.1
            self.stable_supply += mint_amount
            self.stable_price = max(self.peg, self.stable_price - mint_amount / self.stable_supply)
            
        elif self.stable_price < self.peg * 0.995:  # Below peg
            # Mint equity tokens to buy back stablecoins
            deficit = (self.peg - self.stable_price) * self.stable_supply
            buy_amount = deficit * 0.1 * confidence  # Scaled by confidence
            
            if self.equity_price > 0.01:
                equity_to_mint = buy_amount / self.equity_price
                self.equity_supply += equity_to_mint
                
                # Equity price drops due to dilution
                dilution = equity_to_mint / self.equity_supply
                self.equity_price *= (1 - dilution) * confidence
                self.equity_price = max(0.001, self.equity_price)
                
                # Buy back stablecoins
                stable_bought = min(buy_amount, self.stable_supply * 0.05)
                self.stable_supply -= stable_bought
                self.stable_price += stable_bought / self.stable_supply * 0.5
        
        self.equity_mcap = self.equity_supply * self.equity_price
        self._record()


# Scenario 1: Normal operation (small shocks, high confidence)
print("=" * 60)
print("ALGORITHMIC STABLECOIN SIMULATION")
print("=" * 60)

algo_normal = AlgorithmicStablecoin()
np.random.seed(42)
for _ in range(100):
    shock = np.random.normal(0, 50_000_000)  # Small random shocks
    algo_normal.step(shock, confidence=0.95)

print(f"\nNormal Operation (100 steps):")
print(f"  Stable price range: ${min(h['stable_price'] for h in algo_normal.history):.4f} - "
      f"${max(h['stable_price'] for h in algo_normal.history):.4f}")
print(f"  Final stable price: ${algo_normal.stable_price:.4f}")
print(f"  Equity price: ${algo_normal.equity_price:.2f}")

In [ ]:
# Scenario 2: Death spiral (large sell-off, collapsing confidence)
algo_death = AlgorithmicStablecoin()
np.random.seed(42)

# Phase 1: Normal operation (20 steps)
for _ in range(20):
    algo_death.step(np.random.normal(0, 50_000_000), confidence=0.95)

# Phase 2: Large sell-off begins
print("\n--- DEATH SPIRAL SIMULATION (Terra/LUNA-style) ---")
for i in range(80):
    # Increasing selling pressure
    sell_pressure = -200_000_000 * (1 + i * 0.1)
    # Confidence drops as depeg worsens
    depeg = abs(algo_death.stable_price - 1.0)
    confidence = max(0.05, 1.0 - depeg * 5)
    algo_death.step(sell_pressure, confidence=confidence)

print(f"  Final stable price: ${algo_death.stable_price:.4f}")
print(f"  Final equity price: ${algo_death.equity_price:.4f}")
print(f"  Equity dilution: {algo_death.equity_supply / 1e9:.1f}B tokens (started at 1B)")
print(f"  Stable MCap destroyed: ${(10e9 - algo_death.stable_supply * algo_death.stable_price) / 1e9:.1f}B")

# Visualize both scenarios
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Normal operation
normal_stable = [h['stable_price'] for h in algo_normal.history]
axes[0, 0].plot(normal_stable, 'g-', linewidth=2)
axes[0, 0].axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
axes[0, 0].set_title('Normal Operation: Stable Price')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].set_ylim(0.95, 1.05)

# Death spiral: stable price
death_stable = [h['stable_price'] for h in algo_death.history]
axes[0, 1].plot(death_stable, 'r-', linewidth=2)
axes[0, 1].axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
axes[0, 1].set_title('Death Spiral: Stable Price')
axes[0, 1].set_ylabel('Price ($)')

# Death spiral: equity price
death_equity = [h['equity_price'] for h in algo_death.history]
axes[1, 0].plot(death_equity, 'purple', linewidth=2)
axes[1, 0].set_title('Death Spiral: Equity Token Price')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].set_xlabel('Time Step')
axes[1, 0].set_yscale('log')

# Death spiral: equity supply (hyperinflation)
death_supply = [h['equity_supply'] / 1e9 for h in algo_death.history]
axes[1, 1].plot(death_supply, 'orange', linewidth=2)
axes[1, 1].set_title('Death Spiral: Equity Token Supply')
axes[1, 1].set_ylabel('Supply (Billions)')
axes[1, 1].set_xlabel('Time Step')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig('/tmp/death_spiral.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nThe death spiral shows hyperinflationary equity token minting")
print("that fails to restore the peg -- exactly what happened to Terra/LUNA.")

---
## 5. Stablecoin Risk Metrics

Key metrics for evaluating stablecoin health and risk:

- **Peg Deviation**: How closely the stablecoin tracks its target price
- **Collateralization Health**: For crypto-backed, the ratio of collateral to outstanding debt
- **Concentration Risk**: How diversified the collateral/reserves are
- **Redemption Capacity**: Can the system honor all redemptions simultaneously?

> **Definition: Basis Points (bps)** -- One hundredth of a percentage point (0.01%). Used to express small price deviations. 100 bps = 1%.

In [ ]:
def peg_deviation_metrics(prices: np.ndarray, peg: float = 1.0) -> Dict[str, float]:
    """Calculate peg stability metrics."""
    deviations = prices - peg
    abs_deviations = np.abs(deviations)
    
    return {
        'mean_deviation_bps': np.mean(abs_deviations) * 10000,
        'max_deviation_bps': np.max(abs_deviations) * 10000,
        'std_deviation_bps': np.std(deviations) * 10000,
        'pct_within_50bps': np.mean(abs_deviations < 0.005) * 100,
        'pct_within_100bps': np.mean(abs_deviations < 0.01) * 100,
        'max_above_peg': np.max(deviations) * 100,
        'max_below_peg': np.min(deviations) * 100,
        'time_below_peg_pct': np.mean(deviations < 0) * 100,
    }


# Generate synthetic price data for multiple stablecoins
np.random.seed(42)
days = 365

stable_prices = {
    'USDC': 1.0 + np.random.normal(0, 0.0005, days),  # Very tight peg
    'DAI': 1.0 + np.random.normal(0, 0.002, days),      # Moderate deviation
    'FRAX': 1.0 + np.random.normal(0, 0.001, days),     # Moderate-tight
    'LUSD': 1.0 + np.random.normal(0.001, 0.003, days), # Slight premium
}

# Add a depeg event to USDC (SVB crisis, March 2023)
stable_prices['USDC'][70:75] -= np.array([0.02, 0.05, 0.08, 0.06, 0.02])

print("=" * 80)
print("STABLECOIN PEG STABILITY METRICS")
print("=" * 80)

print(f"\n{'Coin':<8} {'Mean Dev':>10} {'Max Dev':>10} {'<50bps':>8} {'<100bps':>9} {'Max Depeg':>11}")
print("-" * 60)
for name, prices in stable_prices.items():
    m = peg_deviation_metrics(prices)
    print(f"{name:<8} {m['mean_deviation_bps']:>8.1f}bps {m['max_deviation_bps']:>8.1f}bps "
          f"{m['pct_within_50bps']:>7.1f}% {m['pct_within_100bps']:>8.1f}% "
          f"{m['max_below_peg']:>9.2f}%")

In [ ]:
# Visualize peg stability
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'USDC': 'blue', 'DAI': 'orange', 'FRAX': 'green', 'LUSD': 'purple'}

# Price over time
for name, prices in stable_prices.items():
    axes[0, 0].plot(prices, color=colors[name], alpha=0.7, label=name)
axes[0, 0].axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
axes[0, 0].fill_between(range(days), 0.995, 1.005, alpha=0.1, color='green')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].set_title('Stablecoin Price History')
axes[0, 0].legend(fontsize=8)
axes[0, 0].set_ylim(0.90, 1.05)

# Deviation distribution
for name, prices in stable_prices.items():
    devs = (prices - 1.0) * 10000  # Convert to bps
    axes[0, 1].hist(devs, bins=50, alpha=0.5, color=colors[name], label=name)
axes[0, 1].set_xlabel('Deviation (bps)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Peg Deviation Distribution')
axes[0, 1].legend(fontsize=8)

# Rolling standard deviation
window = 30
for name, prices in stable_prices.items():
    rolling_std = np.array([np.std(prices[max(0,i-window):i+1]) * 10000
                           for i in range(len(prices))])
    axes[1, 0].plot(rolling_std, color=colors[name], label=name, alpha=0.7)
axes[1, 0].set_xlabel('Day')
axes[1, 0].set_ylabel('Rolling 30d Std Dev (bps)')
axes[1, 0].set_title('Peg Volatility Over Time')
axes[1, 0].legend(fontsize=8)

# Risk comparison radar
metrics_list = ['mean_deviation_bps', 'max_deviation_bps', 'std_deviation_bps']
metric_labels = ['Mean Dev', 'Max Dev', 'Std Dev']
coin_names = list(stable_prices.keys())

x = np.arange(len(metric_labels))
width = 0.2
for i, name in enumerate(coin_names):
    m = peg_deviation_metrics(stable_prices[name])
    vals = [m[k] for k in metrics_list]
    axes[1, 1].bar(x + i * width, vals, width, color=colors[name], label=name)

axes[1, 1].set_xticks(x + width * 1.5)
axes[1, 1].set_xticklabels(metric_labels)
axes[1, 1].set_ylabel('Basis Points')
axes[1, 1].set_title('Risk Metrics Comparison')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/stablecoin_risk.png', dpi=100, bbox_inches='tight')
plt.show()
print("Stablecoin risk analysis complete.")

---
## 6. Interest Rates and Yield

Stablecoin yield comes from several sources:

- **Lending rates**: Supplying stablecoins to lending protocols (Aave, Compound)
- **Liquidity provision**: Providing stablecoin liquidity to AMMs (Curve)
- **Savings rates**: Protocol-native rates (e.g., Dai Savings Rate / DSR)
- **RWA (Real-World Asset) yields**: Backed by T-bills or money market funds

> **Definition: Dai Savings Rate (DSR)** -- An interest rate paid to DAI holders who deposit their DAI into the DSR contract. Set by MakerDAO governance, it serves as a monetary policy tool to influence DAI supply and demand.

In [ ]:
# Stablecoin yield comparison
np.random.seed(42)

@dataclass
class YieldSource:
    """A stablecoin yield source."""
    name: str
    stablecoin: str
    base_apy: float
    risk_score: float  # 1-10 (higher = riskier)
    category: str


yield_sources = [
    YieldSource("Aave USDC", "USDC", 0.04, 2, "Lending"),
    YieldSource("Compound USDC", "USDC", 0.035, 2, "Lending"),
    YieldSource("Curve 3pool", "USDC/DAI/USDT", 0.03, 3, "LP"),
    YieldSource("DAI Savings Rate", "DAI", 0.05, 2, "Savings"),
    YieldSource("Ethena sUSDe", "USDe", 0.15, 7, "Synthetic"),
    YieldSource("New DeFi Protocol", "USDC", 0.25, 9, "DeFi"),
    YieldSource("T-Bills (reference)", "USD", 0.05, 1, "TradFi"),
]

print("=" * 70)
print("STABLECOIN YIELD COMPARISON")
print("=" * 70)

print(f"\n{'Source':<22} {'Coin':<15} {'APY':>6} {'Risk':>6} {'Risk-Adj':>10} {'Category':<10}")
print("-" * 72)

for ys in yield_sources:
    # Risk-adjusted yield (simplified Sharpe-like)
    risk_adj = ys.base_apy / (ys.risk_score / 2)
    print(f"{ys.name:<22} {ys.stablecoin:<15} {ys.base_apy:>5.1%} {ys.risk_score:>5}/10 "
          f"{risk_adj:>9.3f} {ys.category:<10}")

print(f"\nRule of thumb: If yield >> risk-free rate, ask WHERE the yield comes from.")
print(f"Unsustainable yields are often funded by token emissions (temporary).")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

risks = [ys.risk_score for ys in yield_sources]
yields_pct = [ys.base_apy * 100 for ys in yield_sources]
names = [ys.name for ys in yield_sources]

cat_colors = {'Lending': 'blue', 'LP': 'green', 'Savings': 'orange',
              'Synthetic': 'purple', 'DeFi': 'red', 'TradFi': 'gray'}
point_colors = [cat_colors[ys.category] for ys in yield_sources]

ax.scatter(risks, yields_pct, s=150, c=point_colors, zorder=5)
for i, name in enumerate(names):
    ax.annotate(name, (risks[i], yields_pct[i]), fontsize=8,
                xytext=(8, 4), textcoords='offset points')

ax.set_xlabel('Risk Score (1-10)')
ax.set_ylabel('APY (%)')
ax.set_title('Stablecoin Yield vs Risk')

# Add risk-free rate line
ax.axhline(y=5, color='gray', linestyle='--', alpha=0.5, label='Risk-free rate (~5%)')
ax.legend()

plt.tight_layout()
plt.savefig('/tmp/stablecoin_yield.png', dpi=100, bbox_inches='tight')
plt.show()
print("Higher yields come with proportionally higher risks.")

---
## 7. Reserve Composition and Regulatory Framework

The quality and transparency of stablecoin reserves directly impacts systemic risk. Different jurisdictions are developing regulatory frameworks for stablecoins.

Key regulatory considerations:
- **Reserve requirements**: What assets can back stablecoins? (Cash, T-bills, commercial paper)
- **Attestation/audit frequency**: How often are reserves verified?
- **Redemption rights**: Can holders redeem 1:1 at any time?
- **Capital requirements**: Do issuers need minimum capital buffers?

In [ ]:
# Reserve composition analysis
reserve_compositions = {
    "USDC (Circle)": {
        "Cash": 0.15, "US Treasuries": 0.80,
        "Repo Agreements": 0.05
    },
    "USDT (Tether)": {
        "Cash": 0.05, "US Treasuries": 0.65,
        "Corporate Bonds": 0.10, "Secured Loans": 0.08,
        "Other": 0.07, "Digital Tokens": 0.05
    },
    "DAI (Maker)": {
        "ETH": 0.25, "USDC": 0.30,
        "RWA (T-Bills)": 0.35, "Other Crypto": 0.10
    },
    "Ideal (Regulatory)": {
        "Cash": 0.20, "US Treasuries": 0.70,
        "Repo Agreements": 0.10
    },
}

def reserve_quality_score(composition: Dict[str, float]) -> float:
    """Score reserve quality (higher = safer)."""
    quality_weights = {
        "Cash": 1.0, "US Treasuries": 0.95, "Repo Agreements": 0.90,
        "RWA (T-Bills)": 0.90, "USDC": 0.85,
        "Corporate Bonds": 0.70, "Secured Loans": 0.60,
        "ETH": 0.50, "Other Crypto": 0.40,
        "Digital Tokens": 0.30, "Other": 0.50
    }
    score = sum(pct * quality_weights.get(asset, 0.5)
                for asset, pct in composition.items())
    return score * 100


print("=" * 60)
print("RESERVE COMPOSITION ANALYSIS")
print("=" * 60)

for issuer, comp in reserve_compositions.items():
    score = reserve_quality_score(comp)
    print(f"\n{issuer} (Quality Score: {score:.0f}/100)")
    for asset, pct in sorted(comp.items(), key=lambda x: -x[1]):
        bar = '█' * int(pct * 40)
        print(f"  {asset:<20} {pct:>5.0%} {bar}")

# Visualize as stacked bar
fig, ax = plt.subplots(figsize=(12, 6))

issuers = list(reserve_compositions.keys())
all_assets = set()
for comp in reserve_compositions.values():
    all_assets.update(comp.keys())
all_assets = sorted(all_assets)

asset_colors = plt.cm.Set3(np.linspace(0, 1, len(all_assets)))
bottom = np.zeros(len(issuers))

for j, asset in enumerate(all_assets):
    values = [reserve_compositions[i].get(asset, 0) * 100 for i in issuers]
    ax.barh(issuers, values, left=bottom, color=asset_colors[j], label=asset)
    bottom += values

ax.set_xlabel('Allocation (%)')
ax.set_title('Stablecoin Reserve Composition')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/reserve_composition.png', dpi=100, bbox_inches='tight')
plt.show()
print("Reserve quality varies significantly between issuers.")

---
## Exercises

### Exercise 1: Stablecoin Stress Tester

Build a comprehensive stress testing framework for different stablecoin types. Test scenarios like bank runs, collateral crashes, and oracle failures.

**Hints:**
- Model different redemption scenarios (gradual vs panic)
- For crypto-collateralized: simulate rapid collateral price drops
- For algorithmic: simulate confidence shocks
- Track how each type performs under identical stress

In [ ]:
class StablecoinStressTester:
    """Stress testing framework for stablecoins."""
    
    def __init__(self, stablecoin_type: str, market_cap: float,
                 collateral_ratio: float = 1.0) -> None:
        """Initialize stress tester."""
        self.type = stablecoin_type
        self.market_cap = market_cap
        self.collateral_ratio = collateral_ratio
        # YOUR CODE HERE
    
    def bank_run(self, redemption_pct: float, duration_days: int) -> Dict:
        """Simulate a bank run with given redemption rate."""
        # YOUR CODE HERE
        pass
    
    def collateral_crash(self, price_drop_pct: float) -> Dict:
        """Simulate collateral price crash (crypto-collateralized only)."""
        # YOUR CODE HERE
        pass
    
    def oracle_failure(self, stale_duration_hours: int) -> Dict:
        """Simulate oracle price feed failure."""
        # YOUR CODE HERE
        pass

### Exercise 2: Multi-Collateral CDP System

Extend the CDP system to support multiple collateral types with different risk parameters (like MakerDAO's multi-collateral DAI).

**Hints:**
- Each collateral type has its own liquidation ratio, stability fee, and debt ceiling
- System-wide metrics aggregate across all collateral types
- Correlation between collateral prices affects systemic risk

In [ ]:
class MultiCollateralCDP:
    """Multi-collateral CDP system."""
    
    def __init__(self) -> None:
        """Initialize multi-collateral system."""
        self.collateral_types: Dict[str, Dict] = {}
        # YOUR CODE HERE
    
    def add_collateral_type(self, name: str, liquidation_ratio: float,
                            stability_fee: float, debt_ceiling: float,
                            initial_price: float) -> None:
        """Add a supported collateral type."""
        # YOUR CODE HERE
        pass
    
    def open_vault(self, owner: str, collateral_type: str,
                   amount: float, debt: float) -> int:
        """Open a vault with specified collateral."""
        # YOUR CODE HERE
        pass
    
    def system_risk_report(self) -> Dict:
        """Generate a comprehensive risk report."""
        # YOUR CODE HERE
        pass

### Exercise 3: Stablecoin Portfolio Optimizer

Build a portfolio optimizer that allocates across multiple stablecoins to minimize depeg risk while maximizing yield.

**Hints:**
- Different stablecoins have different depeg risks (correlated and uncorrelated)
- USDC depegged during SVB crisis -- bank risk
- DAI is partially backed by USDC -- contagion risk
- Diversification reduces but doesn't eliminate risk

In [ ]:
class StablecoinPortfolio:
    """Optimized stablecoin portfolio."""
    
    def __init__(self, stablecoins: Dict[str, Dict]) -> None:
        """Initialize with stablecoin profiles.
        
        stablecoins: {name: {yield, risk, correlation_group}}
        """
        self.stablecoins = stablecoins
        # YOUR CODE HERE
    
    def optimize(self, target: str = 'risk') -> Dict[str, float]:
        """Find optimal allocation.
        
        target: 'risk' (minimize risk) or 'sharpe' (maximize risk-adj yield)
        """
        # YOUR CODE HERE
        pass
    
    def stress_test(self, allocation: Dict[str, float],
                    scenario: str) -> Dict:
        """Stress test a portfolio allocation."""
        # YOUR CODE HERE
        pass

---
## Summary

### What You Learned
- [x] Three stablecoin architectures: fiat-collateralized, crypto-collateralized, algorithmic
- [x] Peg arbitrage mechanics and how arbitrage strength affects peg stability
- [x] CDP system mechanics: collateralization, stability fees, liquidation cascades
- [x] Algorithmic stablecoin mint/burn dynamics and the death spiral failure mode
- [x] Stablecoin risk metrics: peg deviation, collateralization health, concentration risk
- [x] Yield sources for stablecoins and risk-adjusted return comparison
- [x] Reserve composition analysis and quality scoring

### Key Takeaways
1. **No stablecoin is truly risk-free** -- each design makes different tradeoffs between decentralization, capital efficiency, and peg stability
2. **Crypto-collateralized stablecoins require over-collateralization** -- the safety margin is the cost of decentralization
3. **Algorithmic stablecoins are inherently fragile** -- they work in good times but can death-spiral when confidence breaks
4. **Reserve quality matters enormously** -- cash and T-bills are far safer than corporate bonds or crypto
5. **High yields on stablecoins signal high risk** -- if yield >> risk-free rate, understand the source
6. **Contagion risk exists between stablecoins** -- DAI backed by USDC means USDC risk propagates

### Further Reading
- Moin, A. et al. (2020). "SoK: A Classification Framework for Stablecoin Designs." FC 2020.
- Clements, R. (2021). "Built to Fail: The Inherent Fragility of Algorithmic Stablecoins."
- MakerDAO. (2023). "Maker Protocol Documentation." docs.makerdao.com

### Next Steps
- [Notebook 14: Consensus Simulations](14-consensus-simulations.ipynb) -- Byzantine fault tolerance and consensus mechanisms
- [Section 07: Stablecoins](../sections/07-stablecoins.md) -- Comprehensive stablecoin theory